# 06 XGBoost Modelling and Prediction Export

This notebook trains the XGBoost model, evaluates utilisation and bike-level performance, then exports a **threshold-neutral** holdout prediction file for downstream business-threshold analysis.

In [1]:
import torch
import time


print("torch:", torch.__version__)
print("cuda runtime:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("archs:", torch.cuda.get_arch_list())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

x = torch.rand(3000, 3000, device=device)
y = torch.rand(3000, 3000, device=device)

torch.cuda.synchronize()
t0 = time.time()
z = x @ y
torch.cuda.synchronize()
t1 = time.time()

print("Result device:", z.device)
print("Elapsed:", t1 - t0, "seconds")

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED) 
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, f1_score

torch: 2.10.0+cu128
cuda runtime: 12.8
cuda available: True
archs: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
gpu: NVIDIA GeForce RTX 5060 Laptop GPU
Using: cuda
Result device: cuda:0
Elapsed: 0.2238450050354004 seconds


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, f1_score


from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    classification_report,
    roc_auc_score
)


import matplotlib.pyplot as plt
import seaborn as sns


## 1. Load engineered dataset with trend

In [3]:
PROJECT_ROOT = Path('..').resolve()
FEATURES_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dublin_bikes_features_v2.parquet'
if not FEATURES_PATH.exists():
    FEATURES_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dublin_bikes_features_v2.csv.gz'

STATIONS_FEATURES_V2_PATH = PROJECT_ROOT / 'data' / 'processed' / 'stations_features_v2.parquet'
if not STATIONS_FEATURES_V2_PATH.exists():
    raise FileNotFoundError(f"Missing station master file: {STATIONS_FEATURES_V2_PATH}")

if FEATURES_PATH.suffix == '.parquet':
    df = pd.read_parquet(FEATURES_PATH).copy()
else:
    df = pd.read_csv(FEATURES_PATH).copy()

stations_features_v2 = pd.read_parquet(STATIONS_FEATURES_V2_PATH).copy()
print(stations_features_v2.columns)
# Keep latest station-level attributes, including the new business capacity column
station_cols = [
    c for c in [
        "station_id",
        "capacity",
        "capacity_non_unique",
        "n_unique_capacity",
        "first_capacity_change",
        "last_capacity_change",
        "name",
        "station_name",
        "station_name_clean",
    ]
    if c in stations_features_v2.columns
]

df = df.merge(
    stations_features_v2[station_cols].drop_duplicates(subset=["station_id"]),
    on="station_id",
    how="left",
    validate="m:1",
)

df = df.sort_values(['station_id', 'last_reported'])
print(df.shape)
print("Added columns from stations_features_v2:", [c for c in station_cols if c != "station_id"])
print("Missing capacity values:", df["capacity"].isna().sum())
df.head()


Index(['station_id', 'lat', 'lon', 'distance_center', 'center_index',
       'area_type_str', 'tourism', 'capacity', 'n_unique_capacity',
       'first_capacity_change', 'last_capacity_change', 'capacity_non_unique'],
      dtype='str')
(13513883, 24)
Added columns from stations_features_v2: ['capacity', 'capacity_non_unique', 'n_unique_capacity', 'first_capacity_change', 'last_capacity_change']
Missing capacity values: 0


,station_id,name,last_reported,hour_slot,hour_sin,hour_cos,days_of_week,is_weekday,roll_mean_30min,roll_std_30min,...,time_slot_str,empty_risk,full_risk,center_index,area_type_str,capacity,capacity_non_unique,n_unique_capacity,first_capacity_change,last_capacity_change
0,1,CLARENDON ROW,2022-04-27 13:00:02,13,-0.258819,-0.965926,2,True,0.0,0.0,...,13:00,True,False,1,retail,31,False,1,NaT,NaT
1,1,CLARENDON ROW,2022-04-27 13:30:02,13,-0.382683,-0.923880,2,True,0.0,0.0,...,13:30,True,False,1,retail,31,False,1,NaT,NaT
2,1,CLARENDON ROW,2022-04-27 14:00:02,14,-0.500000,-0.866025,2,True,0.0,0.0,...,14:00,True,False,1,retail,31,False,1,NaT,NaT
3,1,CLARENDON ROW,2022-04-27 14:30:02,14,-0.608761,-0.793353,2,True,0.0,0.0,...,14:30,True,False,1,retail,31,False,1,NaT,NaT
4,1,CLARENDON ROW,2022-04-27 15:00:02,15,-0.707107,-0.707107,2,True,0.0,0.0,...,15:00,True,False,1,retail,31,False,1,NaT,NaT


## 2. Create targets


Due to irregular sampling intervals, the target variable was defined as the utilisation at the first available observation occurring at least 30 minutes after the current timestamp. This introduces slight variability in the prediction horizon but avoids interpolation artifacts.

In [4]:
# target variable: utilisation 30 min in the future

df = df.sort_values(["station_id", "last_reported"]).copy()

def get_target(group):
    group = group.sort_values("last_reported").copy()

    # preserve station_id explicitly
    sid = group["station_id"].iloc[0]

    times = group["last_reported"].to_numpy()
    values = group["utilisation"].to_numpy()

    target = np.full(len(group), np.nan)

    for i in range(len(times)):
        t_future = times[i] + np.timedelta64(30, "m")
        idx = np.searchsorted(times, t_future, side="left")
        if idx < len(times):
            target[i] = values[idx]

    group["utilisation_tplus_30"] = target
    group["station_id"] = sid
    return group

parts = []
for _, g in df.groupby("station_id", sort=False):
    parts.append(get_target(g))

df = pd.concat(parts, ignore_index=True)
print(df.shape)
print(df.columns)



  

(13513883, 25)
Index(['station_id', 'name', 'last_reported', 'hour_slot', 'hour_sin',
       'hour_cos', 'days_of_week', 'is_weekday', 'roll_mean_30min',
       'roll_std_30min', 'utilisation', 'dock_ratio', 'util_slope',
       'util_vs_mean', 'time_slot_str', 'empty_risk', 'full_risk',
       'center_index', 'area_type_str', 'capacity', 'capacity_non_unique',
       'n_unique_capacity', 'first_capacity_change', 'last_capacity_change',
       'utilisation_tplus_30'],
      dtype='str')


In [5]:
# df.info()

print(df.columns)
print(df.index.names)

Index(['station_id', 'name', 'last_reported', 'hour_slot', 'hour_sin',
       'hour_cos', 'days_of_week', 'is_weekday', 'roll_mean_30min',
       'roll_std_30min', 'utilisation', 'dock_ratio', 'util_slope',
       'util_vs_mean', 'time_slot_str', 'empty_risk', 'full_risk',
       'center_index', 'area_type_str', 'capacity', 'capacity_non_unique',
       'n_unique_capacity', 'first_capacity_change', 'last_capacity_change',
       'utilisation_tplus_30'],
      dtype='str')
[None]


## 3. Drop missing rows

The dataset is not missing many observations over 30 minutes intervalls: the worse station has only 0.0126% missing rows

count    115.000000 stations

mean       0.000033 

std        0.000010 

min        0.000022 

25%        0.000027

50%        0.000033

75%        0.000035

max        0.000126



In [6]:
# Count na per station
na_per_station = (
    df.groupby("station_id")["utilisation_tplus_30"]
      .apply(lambda x: x.isna().sum())
      .reset_index(name="na_count")
)

# Add total rows per station
counts = (
    df.groupby("station_id")["utilisation_tplus_30"]
      .size()
      .reset_index(name="total_rows")
)

na_stats = na_per_station.merge(counts, on="station_id")

# Percentage of na 
na_stats["na_ratio"] = na_stats["na_count"] / na_stats["total_rows"]

# worst stations with most na in target
na_stats.sort_values("na_ratio", ascending=False).head(10)
problematic = na_stats[na_stats["na_ratio"] > 0.5]
print(problematic)
print(na_stats["na_ratio"].describe())


# Drop rows with na in target
df = df.dropna(subset=["utilisation_tplus_30", "util_slope", "util_vs_mean"])


Empty DataFrame
Columns: [station_id, na_count, total_rows, na_ratio]
Index: []
count    115.000000
mean       0.000033
std        0.000010
min        0.000022
25%        0.000027
50%        0.000033
75%        0.000035
max        0.000126
Name: na_ratio, dtype: float64


## 4. Feature list and target definition

In [7]:
# Features

features = [
    "utilisation",
    # "roll_mean_30min", #remove due to high correlation with utilisation
    "roll_std_30min",
    # "dock_ratio", #remove due to high correlation with utilisation
    # "hour_slot",
    "hour_sin", 
    "hour_cos",
    "is_weekday",
    "center_index",
    'area_type_str',
    "util_slope", # trend of utilisation in last 30 min
    "util_vs_mean", # utilisation vs mean of station in last 30 min
    # "station_cluster"   # if added after Maxime's clustering
]
# One‑hot encode area type and center index

X = df[features]
X = pd.get_dummies(X, columns=['area_type_str', "center_index"], drop_first=True)

y = df["utilisation_tplus_30"]
print("Target variable:")
print(y.describe())
print("Missing values in target:", y.isna().sum())


Target variable:
count    1.350982e+07
mean     3.800239e-01
std      3.047142e-01
min      0.000000e+00
25%      1.000000e-01
50%      3.250000e-01
75%      6.000000e-01
max      1.000000e+00
Name: utilisation_tplus_30, dtype: float64
Missing values in target: 0


## 6. Train/test split

In [8]:

# Time‑based split: last 365 days as test
cutoff = df['last_reported'].max() - pd.Timedelta(days=365)
train_mask = df['last_reported'] < cutoff

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]


In [9]:
print("NaN in y_test:", pd.isna(y_test).sum())

print("NaN in X_train:", np.isnan(X_train).sum())
print("NaN in X_test:", np.isnan(X_test).sum())

print("Inf in X_train:", np.isinf(X_train).sum())
print("Inf in X_test:", np.isinf(X_test).sum())

NaN in y_test: 0
NaN in X_train: utilisation             0
roll_std_30min          0
hour_sin                0
hour_cos                0
is_weekday              0
util_slope              0
util_vs_mean            0
area_type_str_retail    0
area_type_str_work      0
center_index_2          0
center_index_3          0
dtype: int64
NaN in X_test: utilisation             0
roll_std_30min          0
hour_sin                0
hour_cos                0
is_weekday              0
util_slope              0
util_vs_mean            0
area_type_str_retail    0
area_type_str_work      0
center_index_2          0
center_index_3          0
dtype: int64
Inf in X_train: utilisation             0
roll_std_30min          0
hour_sin                0
hour_cos                0
is_weekday              0
util_slope              0
util_vs_mean            0
area_type_str_retail    0
area_type_str_work      0
center_index_2          0
center_index_3          0
dtype: int64
Inf in X_test: utilisation             

## 7. XGBoost Regression

In [10]:
xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)


# Predictions

y_pred_xgb = xgb_model.predict(X_test)


# Evaluation
mae = mean_absolute_error(y_test, y_pred_xgb)
rmse = root_mean_squared_error(y_test, y_pred_xgb)

print(f"XGBoost MAE: {mae:.4f}")
print(f"XGBoost RMSE: {rmse:.4f}")

# Conversion to empty/full bands
EMPTY_THRESHOLD = 0.05
FULL_THRESHOLD  = 0.95

empty_pred_xgb = (y_pred_xgb < EMPTY_THRESHOLD).astype(int)
full_pred_xgb  = (y_pred_xgb > FULL_THRESHOLD).astype(int)

empty_true = (y_test < EMPTY_THRESHOLD).astype(int)
full_true  = (y_test > FULL_THRESHOLD).astype(int)


# Evaluation

print("=== XGBOOST - EMPTY ===")
print(classification_report(empty_true, empty_pred_xgb))

print("=== XGBOOST - FULL ===")
print(classification_report(full_true, full_pred_xgb))

print("EMPTY ROC AUC:", roc_auc_score(empty_true, -y_pred_xgb))
print("FULL ROC AUC:", roc_auc_score(full_true, y_pred_xgb))

# Coefficients inspection
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": xgb_model.feature_importances_
}).sort_values("coefficient", key=abs, ascending=False) 
print("Top coefficients:")
print(coef_df.head(20).to_string(index=False))


XGBoost MAE: 0.0416
XGBoost RMSE: 0.0744
=== XGBOOST - EMPTY ===
              precision    recall  f1-score   support

           0       0.96      0.97      0.97   5734721
           1       0.83      0.78      0.81   1047906

    accuracy                           0.94   6782627
   macro avg       0.90      0.88      0.89   6782627
weighted avg       0.94      0.94      0.94   6782627

=== XGBOOST - FULL ===
              precision    recall  f1-score   support

           0       0.98      0.99      0.99   6432061
           1       0.85      0.55      0.67    350566

    accuracy                           0.97   6782627
   macro avg       0.91      0.77      0.83   6782627
weighted avg       0.97      0.97      0.97   6782627

EMPTY ROC AUC: 0.9720922211546948
FULL ROC AUC: 0.9831181233619463
Top coefficients:
             feature  coefficient
         utilisation     0.879127
area_type_str_retail     0.049829
      center_index_2     0.019327
          is_weekday     0.013453
   

## 8. Business translation: predicted bikes

This section converts predicted future utilisation into **predicted number of bikes** using station capacity.

The goal for submission is intentionally simple:
- convert utilisation predictions into bikes
- compute bike deficit and bike surplus
- create simple empty/full risk flags
- export clean business-facing tables

In [11]:
# Build the holdout business view from the test rows
df_test = df.loc[~train_mask].copy()

def clip_bikes(values, capacity):
    return values.clip(lower=0, upper=capacity).round().astype(int)

if "capacity" not in df_test.columns:
    raise ValueError("Column 'capacity' is missing. Make sure stations_features_v2.parquet was merged into df.")

if df_test["capacity"].isna().any():
    missing_stations = (
        df_test.loc[df_test["capacity"].isna(), "station_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )
    raise ValueError(
        f"Missing capacity values for stations: {missing_stations[:10]}"
        + (" ..." if len(missing_stations) > 10 else "")
    )

print("Using station capacity from stations_features_v2.parquet")

# Keep model outputs
df_test["pred_utilisation_tplus_30"] = y_pred_xgb
df_test["true_utilisation_tplus_30"] = y_test.to_numpy()

# Convert utilisation to bikes
df_test["current_bikes"] = clip_bikes(df_test["utilisation"] * df_test["capacity"], df_test["capacity"])
df_test["true_bikes_tplus_30"] = clip_bikes(df_test["true_utilisation_tplus_30"] * df_test["capacity"], df_test["capacity"])
df_test["pred_bikes_tplus_30"] = clip_bikes(df_test["pred_utilisation_tplus_30"] * df_test["capacity"], df_test["capacity"])

df_test["true_free_docks_tplus_30"] = df_test["capacity"] - df_test["true_bikes_tplus_30"]
df_test["pred_free_docks_tplus_30"] = df_test["capacity"] - df_test["pred_bikes_tplus_30"]

# Time fields used in the business tables
df_test["last_reported"] = pd.to_datetime(df_test["last_reported"])
df_test["hour"] = df_test["last_reported"].dt.hour
df_test["month"] = df_test["last_reported"].dt.month
df_test["day"] = df_test["last_reported"].dt.day
df_test["weekday"] = df_test["last_reported"].dt.day_name()
df_test["time"] = df_test["last_reported"].dt.strftime("%H:%M")

display(
    df_test[
        ["station_id", "last_reported", "capacity", "current_bikes", "pred_bikes_tplus_30", "pred_free_docks_tplus_30"]
    ].head()
)

Using station capacity from stations_features_v2.parquet


,station_id,last_reported,capacity,current_bikes,pred_bikes_tplus_30,pred_free_docks_tplus_30
55481,1,2024-08-31 23:25:00,31,25,24,7
55482,1,2024-08-31 23:30:00,31,25,24,7
55483,1,2024-08-31 23:40:00,31,25,25,6
55484,1,2024-08-31 23:50:00,31,25,25,6
55485,1,2024-09-01 00:10:00,31,25,25,6


## 9. Bike-based evaluation

This evaluates the model in terms of **number of bikes**, which is easier to interpret operationally than utilisation ratios.

In [12]:
true_bikes = df_test["true_bikes_tplus_30"]
pred_bikes = df_test["pred_bikes_tplus_30"]

bike_mae = mean_absolute_error(true_bikes, pred_bikes)
bike_rmse = root_mean_squared_error(true_bikes, pred_bikes)

print(f"Bike MAE : {bike_mae:.2f} bikes")
print(f"Bike RMSE: {bike_rmse:.2f} bikes")

Bike MAE : 1.22 bikes
Bike RMSE: 2.28 bikes


## 10. Export threshold-neutral holdout predictions

This export is intentionally **threshold-neutral**. It contains the model predictions and station context needed for downstream business analysis, but does **not** store threshold-specific risk labels or deficit/surplus columns.

Use the next notebook to test scenarios such as 5/5, 3/3, 2/2, or 3/4 without retraining the model.


In [13]:

name_candidates = ["name", "station_name", "station_name_clean"]
name_col = next((c for c in name_candidates if c in df_test.columns), None)

business_cols = [
    "station_id",
    "last_reported",
    "hour",
    "month",
    "day",
    "weekday",
    "time",
    "capacity",
    "utilisation",
    "current_bikes",
    "true_utilisation_tplus_30",
    "pred_utilisation_tplus_30",
    "true_bikes_tplus_30",
    "pred_bikes_tplus_30",
    "true_free_docks_tplus_30",
    "pred_free_docks_tplus_30",
]

if name_col is not None:
    business_cols.insert(1, name_col)

business_view = df_test[business_cols].copy()

export_path = PROJECT_ROOT / "data" / "processed" / "xgb_business_view_threshold_neutral.parquet"
business_view.to_parquet(export_path, index=False)

print(f"Saved threshold-neutral business view to: {export_path}")
print("Columns:")
print(list(business_view.columns))
print("Shape:", business_view.shape)


Saved threshold-neutral business view to: /home/tbeliere/Projects/DCU/2026-csc1144-dublin-bikes/data/processed/xgb_business_view_threshold_neutral.parquet
Columns:
['station_id', 'name', 'last_reported', 'hour', 'month', 'day', 'weekday', 'time', 'capacity', 'utilisation', 'current_bikes', 'true_utilisation_tplus_30', 'pred_utilisation_tplus_30', 'true_bikes_tplus_30', 'pred_bikes_tplus_30', 'true_free_docks_tplus_30', 'pred_free_docks_tplus_30']
Shape: (6782627, 17)
